# Notebook 01b — Raster-to-Grid Aggregation

## README

### Overview

This notebook aggregates **raster files** to the **1 km × 1 km city grid** level, producing one **wide-format panel table per city** where each row represents a unique `(city, spatial_id, date)` combination.

The user can choose to aggregate either:
- **Processed (filled) files** from `01a_pixel_filling.ipynb` — cleaned, gap-filled rasters (default)
- **Raw files** — unprocessed rasters before gap-filling

Set `USE_FILLED_FILES = True` in USER INPUTS to use processed files, or `False` to use raw files.

Aggregation uses **`exactextract`** coverage-weighted zonal statistics, which accounts for partial pixel–polygon overlap at grid boundaries — more accurate than a simple pixel-centre approach.

### Inputs (per city)

| Use case | File location | Source notebook | File pattern |
|---|---|---|---|
| Processed (default) | `data/processed/{city}/{variable}/` | `01a_pixel_filling.ipynb` | `*{variable}*_filled.tif` |
| Raw | `data/raw/{city}/{variable}/` | — | `*{variable}*.tif` |
| Grid | `data/processed/{city}/grid/` | `00a_city_boundary_grid_generation.ipynb` | `*.gpkg` |

### Outputs (per city)

```
data/processed/{city}/grid_panel/
  {city}_grid_panel.csv.gz      ← main panel (city, spatial_id, date, variables)
  {city}_grid_geometry.gpkg     ← spatial_id → geometry lookup (static, no dates)
  checkpoints/                  ← per-date CSVs for resume support
    {city}_{variable}_YYYY-MM-DD.csv
```

### Output schema

```
city | spatial_id | date | variable_1 | variable_2 | ...
```

Geometry is stored separately in `{city}_grid_geometry.gpkg` and can be broadcast back via a single merge on `spatial_id`.

### Pipeline position

```
01_satellite_data_cleaning.ipynb
01a_pixel_filling.ipynb
  └─► 01b_raster_to_grid_aggregation.ipynb  ◄── YOU ARE HERE
        └─► 02_panel_assembly.ipynb
```

### Design principles

- **City-agnostic**: all city-specific settings live in the USER INPUTS cell.
- **Variable-agnostic**: variables are declared as `VariableConfig` dataclasses.
- **Flexible data source**: choose between processed (filled) or raw files via `USE_FILLED_FILES`.
- **Resumable**: per-date checkpoint CSVs allow re-runs to skip completed work.
- **FAST_DEV_MODE**: limits to the first N dates for fast iteration.
- **UTM CRS**: city-specific UTM used throughout to avoid Web Mercator distortion.
- **Memory-efficient export**: dtypes are downcast before saving (float64→float32, object→category).


## Part 0 — Environment Setup

In [ ]:
# Install exactextract if not already present
import importlib
if importlib.util.find_spec("exactextract") is None:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "exactextract", "-q"])
    print("exactextract installed.")
else:
    print("exactextract already available.")

In [ ]:
import sys
import re
import warnings
from dataclasses import dataclass
from pathlib import Path
from typing import List, Optional

import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from tqdm.auto import tqdm
from exactextract import exact_extract

# --- src/ path resolution: walk up from cwd until src/ is found ---
_cwd = Path().resolve()
for _p in [_cwd] + list(_cwd.parents):
    if (_p / "src").is_dir():
        sys.path.insert(0, str(_p / "src"))
        print(f"src/ found at: {_p / 'src'}")
        break

try:
    from config import CityConfig, CITY_CRS
    print("CityConfig loaded from src/config.py")
except ImportError:
    print("Warning: could not import CityConfig — UTM auto-detection unavailable.")
    CityConfig = None

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

print("Libraries loaded.")

## Part 1 — USER INPUTS

**Only edit this cell** to configure cities, variables, and run options.

In [ ]:
# =============================================================================
# ██████████████████████   USER INPUTS   ██████████████████████
# =============================================================================

# --- Project root and data directory ---
PROJECT_ROOT = Path().resolve().parent
DATA_ROOT    = PROJECT_ROOT / "data"

# --- Data source ---
# True  → use processed (filled) files from 01a_pixel_filling.ipynb
# False → use raw files (before gap-filling)
USE_FILLED_FILES = False

# --- Cities to process ---
# name        : slug used in file/folder names (lowercase, underscores)
# display_name: human-readable label for plots and logs
# utm_crs     : EPSG code of the city's UTM zone
# UTM CRS is loaded from src/config.py (CITY_CRS dict) — do not hardcode here.
# To add a new city, add it to CITY_CRS in src/config.py and list it below.
# If a city is not found in CITY_CRS, an error is raised at runtime.
CITIES = [
    {"name": "abuja",        "display_name": "Abuja"},
    {"name": "algiers",      "display_name": "Algiers"},
    {"name": "baghdad",      "display_name": "Baghdad"},
    {"name": "buenos_aires", "display_name": "Buenos Aires"},
    {"name": "cairo",        "display_name": "Cairo"},
    {"name": "cape_town",    "display_name": "Cape Town"},
    {"name": "dakar",        "display_name": "Dakar"},
    {"name": "lima",         "display_name": "Lima"},
    {"name": "los_angeles",  "display_name": "Los Angeles"},
    {"name": "madrid",       "display_name": "Madrid"},
    {"name": "mexico_city",  "display_name": "Mexico City"},
    {"name": "mumbai",       "display_name": "Mumbai"},
    {"name": "new_york",     "display_name": "New York"},
    {"name": "santiago",     "display_name": "Santiago"},
    {"name": "yangon",       "display_name": "Yangon"},
]

# Enrich each city dict with its UTM CRS from config.py
for _c in CITIES:
    if _c["name"] not in CITY_CRS:
        raise KeyError(
            f"City '{_c['name']}' not found in CITY_CRS (src/config.py). "
            f"Add it there before proceeding."
        )
    _c["utm_crs"] = CITY_CRS[_c["name"]]

# --- Variables to aggregate ---
# key          : output column name
# folder_name  : subdirectory under data/{source}/{city}/
# file_pattern : glob to match TIFs (must contain YYYY-MM-DD)
# agg_stat     : 'mean' for intensive variables, 'sum' for additive
# nodata_value : fallback sentinel if raster metadata missing
@dataclass
class VariableConfig:
    key: str
    folder_name: str
    file_pattern: str
    agg_stat: str = "mean"
    nodata_value: float = -9999.0

VARIABLES: List[VariableConfig] = [
    VariableConfig(key="no2",       folder_name="no2",       file_pattern="*no2*.tif"),
    VariableConfig(key="era5_temp", folder_name="era5_temp", file_pattern="*era5_temp*.tif"),
    # VariableConfig(key="lst",       folder_name="lst",       file_pattern="*lst*_filled.tif"),
    # VariableConfig(key="ntl",       folder_name="ntl",       file_pattern="*ntl*_filled.tif"),
]

# --- Run options ---
FAST_DEV_MODE      = False  # True → process only first FAST_DEV_N dates per variable
FAST_DEV_N         = 30
SKIP_COMPLETED     = True   # skip (city, variable, date) combos with existing checkpoints
EXACTEXTRACT_STRATEGY = "raster-sequential"  # or "feature-sequential" for lower memory
GRID_ID_COL        = "spatial_id"

# =============================================================================
# END OF USER INPUTS
# =============================================================================

data_source = "processed" if USE_FILLED_FILES else "raw"
print(f"🔧 Data source: {data_source}")
if FAST_DEV_MODE:
    print(f"⚡ FAST_DEV_MODE ON — limiting to {FAST_DEV_N} dates per variable.")
print(f"Cities   : {[c['name'] for c in CITIES]}")
print(f"Variables: {[v.key for v in VARIABLES]}")

## Part 2 — Helper Functions

In [ ]:
_DATE_RE = re.compile(r"(\d{4}-\d{2}-\d{2})")

def extract_date_from_filename(path: Path) -> Optional[str]:
    """Return the first YYYY-MM-DD string found in a filename, or None."""
    m = _DATE_RE.search(path.name)
    return m.group(1) if m else None


def load_grid(city_name: str, utm_crs: Optional[str]) -> gpd.GeoDataFrame:
    """Load the 1 km grid, reproject to UTM, ensure GRID_ID_COL exists."""
    grid_dir = DATA_ROOT / "processed" / city_name / "grid"
    candidates = (list(grid_dir.glob("*.gpkg")) +
                  list(grid_dir.glob("*.geojson")) +
                  list(grid_dir.glob("*.shp")))
    if not candidates:
        raise FileNotFoundError(f"No grid file found in {grid_dir}")
    grid = gpd.read_file(candidates[0])
    print(f"  Loading grid: {candidates[0].name}")

    if GRID_ID_COL not in grid.columns:
        print(f"  Warning: '{GRID_ID_COL}' not found — creating from row index.")
        grid[GRID_ID_COL] = [f"{city_name}_grid_{i:06d}" for i in range(len(grid))]

    if utm_crs:
        grid = grid.to_crs(utm_crs)
        print(f"  Reprojected to {utm_crs}. Cells: {len(grid)}")
    else:
        print(f"  CRS: {grid.crs}. Cells: {len(grid)}")

    return grid[[GRID_ID_COL, "geometry"]]


def list_tif_files(city_name: str, var: VariableConfig) -> List[Path]:
    """Return all TIFs for a city/variable, sorted by date.
    
    Source depends on USE_FILLED_FILES:
      True  → data/processed/{city}/{var.folder_name}/ (filled files from 01a)
      False → data/raw/{city}/{var.folder_name}/ (raw files before processing)
    """
    source = "processed" if USE_FILLED_FILES else "raw"
    tif_dir = DATA_ROOT / source / city_name / var.folder_name
    if not tif_dir.exists():
        print(f"  Directory not found: {tif_dir}")
        return []
    files = sorted(tif_dir.glob(var.file_pattern))
    return [f for f in files if extract_date_from_filename(f) is not None]


def checkpoint_path(city_name: str, var_key: str, date_str: str) -> Path:
    d = DATA_ROOT / "processed" / city_name / "grid_panel" / "checkpoints"
    d.mkdir(parents=True, exist_ok=True)
    return d / f"{city_name}_{var_key}_{date_str}.csv"

def is_completed(city_name: str, var_key: str, date_str: str) -> bool:
    return SKIP_COMPLETED and checkpoint_path(city_name, var_key, date_str).exists()

def save_checkpoint(df: pd.DataFrame, city_name: str, var_key: str, date_str: str):
    df.to_csv(checkpoint_path(city_name, var_key, date_str), index=False)


def aggregate_one_tif(
    tif_path: Path,
    grid: gpd.GeoDataFrame,
    var: VariableConfig,
    utm_crs: Optional[str],
) -> Optional[pd.DataFrame]:
    """
    Aggregate one TIF to the grid using exactextract coverage-weighted stats.
    Returns DataFrame with columns [GRID_ID_COL, var.key], or None on failure.
    """
    try:
        with rasterio.open(tif_path) as src:
            raster_crs = src.crs
            nodata     = src.nodata if src.nodata is not None else var.nodata_value

        # Reproject grid to raster CRS for exactextract
        grid_for_extract = grid.to_crs(raster_crs) if grid.crs != raster_crs else grid

        result = exact_extract(
            str(tif_path),
            grid_for_extract,
            [var.agg_stat],   # exactextract mean/sum are coverage-weighted by design
            output="pandas",
            strategy=EXACTEXTRACT_STRATEGY,
        )

        # Column name varies by exactextract version — grab whatever came back
        value_col = [c for c in result.columns if c != GRID_ID_COL][0]
        result[GRID_ID_COL] = grid_for_extract[GRID_ID_COL].values
        result = result.rename(columns={value_col: var.key})
        result = result[[GRID_ID_COL, var.key]]

        # Replace nodata sentinels with NaN
        result[var.key] = result[var.key].replace(nodata, np.nan)

        return result

    except Exception as e:
        print(f"    ✗ Error aggregating {tif_path.name}: {e}")
        return None


def downcast_panel(df: pd.DataFrame) -> pd.DataFrame:
    """
    Downcast a panel DataFrame for memory-efficient export.
    - float64  → float32  (halves float memory; sufficient precision for all satellite vars)
    - city/spatial_id → category  (large repeated strings stored as integer codes)
    - date kept as string (already compact in CSV)
    Prints memory reduction summary.
    """
    before_mb = df.memory_usage(deep=True).sum() / 1e6

    df = df.copy()
    for col in df.select_dtypes(include="float64").columns:
        df[col] = df[col].astype("float32")
    for col in ["city", GRID_ID_COL]:
        if col in df.columns:
            df[col] = df[col].astype("category")

    after_mb = df.memory_usage(deep=True).sum() / 1e6
    print(f"  Memory: {before_mb:.1f} MB → {after_mb:.1f} MB "
          f"({100*(1 - after_mb/before_mb):.0f}% reduction)")
    return df


print("Helper functions defined.")

## Part 3 — Pre-flight Checks

Verify inputs exist before running the full aggregation loop.

In [ ]:
print("=" * 60)
print("PRE-FLIGHT CHECKS")
print("=" * 60)

all_ok = True

for city in CITIES:
    city_name = city["name"]
    print(f"\n📍 {city['display_name']} ({city_name})")

    grid_dir   = DATA_ROOT / "processed" / city_name / "grid"
    grid_files = list(grid_dir.glob("*.gpkg")) + list(grid_dir.glob("*.geojson"))
    status     = "✓" if grid_files else "✗ MISSING"
    print(f"  Grid        [{status}]: {grid_dir}")
    if not grid_files:
        all_ok = False

    for var in VARIABLES:
        tif_dir = DATA_ROOT / "raw" / city_name / var.folder_name
        n       = len(list(tif_dir.glob(var.file_pattern))) if tif_dir.exists() else 0
        status  = f"✓ {n} files" if n > 0 else "✗ MISSING or EMPTY"
        print(f"  {var.key:<12} [{status}]: {tif_dir}")
        if n == 0:
            all_ok = False

print()
if all_ok:
    print("✅ All checks passed. Ready to aggregate.")
else:
    print("⚠️  Some inputs are missing — check paths above before proceeding.")

## Part 4 — Pixel-Level Missingness Report

Sample the first 10 filled TIFs per city/variable to confirm gap-filling (`01a`) was applied.

In [ ]:
print(f"{'City':<15} {'Variable':<12} {'Files sampled':>14} {'Mean missing %':>15}")
print("-" * 60)

for city in CITIES:
    for var in VARIABLES:
        files = list_tif_files(city["name"], var)[:10]
        if not files:
            print(f"{city['display_name']:<15} {var.key:<12} {'n/a':>14} {'n/a':>15}")
            continue
        rates = []
        for f in files:
            try:
                with rasterio.open(f) as src:
                    arr = src.read(1).astype(float)
                    nd  = src.nodata
                    if nd is not None:
                        arr[arr == nd] = np.nan
                    rates.append(100 * np.isnan(arr).sum() / arr.size)
            except Exception:
                pass
        mean_pct = np.mean(rates) if rates else float("nan")
        print(f"{city['display_name']:<15} {var.key:<12} {len(files):>14} {mean_pct:>14.1f}%")

print("\nNote: residual NaN after gap-filling propagates to grid cells as NaN.")

## Part 5 — Raster-to-Grid Aggregation

For each city → variable → date: aggregate the filled raster to the 1 km grid.
Each date-slice is checkpointed immediately so the loop can resume if interrupted.

In [ ]:
city_var_frames: dict = {c["name"]: {v.key: [] for v in VARIABLES} for c in CITIES}

for city in CITIES:
    city_name    = city["name"]
    display_name = city["display_name"]
    utm_crs      = city["utm_crs"]

    print("\n" + "=" * 60)
    print(f"Processing: {display_name} ({city_name})")
    print("=" * 60)

    try:
        grid = load_grid(city_name, utm_crs)
    except FileNotFoundError as e:
        print(f"  ✗ Skipping — {e}")
        continue

    for var in VARIABLES:
        tif_files = list_tif_files(city_name, var)
        if not tif_files:
            print(f"\n  [{var.key}] No TIF files found — skipping.")
            continue

        if FAST_DEV_MODE:
            tif_files = tif_files[:FAST_DEV_N]

        print(f"\n  [{var.key}] {len(tif_files)} files — aggregating ({var.agg_stat})...")

        for tif_path in tqdm(tif_files, desc=f"{city_name}/{var.key}", unit="file"):
            date_str = extract_date_from_filename(tif_path)
            if date_str is None:
                continue

            if is_completed(city_name, var.key, date_str):
                df_ckpt = pd.read_csv(checkpoint_path(city_name, var.key, date_str))
                city_var_frames[city_name][var.key].append(df_ckpt)
                continue

            row_df = aggregate_one_tif(tif_path, grid, var, utm_crs)
            if row_df is None:
                continue

            row_df.insert(0, "date", date_str)
            row_df.insert(0, "city", city_name)

            save_checkpoint(row_df, city_name, var.key, date_str)
            city_var_frames[city_name][var.key].append(row_df)

        n = len(city_var_frames[city_name][var.key])
        print(f"  [{var.key}] Done — {n} date-slices collected.")

print("\n✅ Aggregation complete.")

## Part 6 — Merge Variables into Wide Panel Table

Concatenate all date-slices per variable, merge on `(city, spatial_id, date)`.
Also saves the `spatial_id → geometry` lookup file (one-time, no dates).

In [ ]:
city_panels: dict = {}

for city in CITIES:
    city_name = city["name"]
    utm_crs   = city["utm_crs"]

    print(f"\n{'='*50}\nBuilding panel: {city['display_name']}")

    try:
        grid = load_grid(city_name, utm_crs)
    except FileNotFoundError as e:
        print(f"  ✗ {e}")
        continue

    # Save spatial_id → geometry lookup once (static reference, no dates)
    out_dir = DATA_ROOT / "processed" / city_name / "grid_panel"
    out_dir.mkdir(parents=True, exist_ok=True)
    geom_path = out_dir / f"{city_name}_grid_geometry.gpkg"
    if not geom_path.exists():
        grid.to_crs("EPSG:4326").to_file(geom_path, driver="GPKG")
        print(f"  Saved geometry lookup: {geom_path.name}")
    else:
        print(f"  Geometry lookup already exists: {geom_path.name}")

    # Concatenate per-variable long tables
    var_long: dict = {}
    for var in VARIABLES:
        frames = city_var_frames[city_name][var.key]
        if not frames:
            print(f"  [{var.key}] No data — skipping.")
            continue
        long_df = pd.concat(frames, ignore_index=True)
        long_df["date"] = pd.to_datetime(long_df["date"])
        var_long[var.key] = long_df
        print(f"  [{var.key}] {len(long_df):,} rows "
              f"({long_df['date'].nunique()} dates × {long_df[GRID_ID_COL].nunique()} cells)")

    if not var_long:
        print("  No variables — skipping city.")
        continue

    # Merge all variables on (city, spatial_id, date)
    merge_keys = ["city", GRID_ID_COL, "date"]
    panel = list(var_long.values())[0]
    for extra_df in list(var_long.values())[1:]:
        panel = panel.merge(
            extra_df[merge_keys + [extra_df.columns[-1]]],
            on=merge_keys, how="outer"
        )

    panel = panel.sort_values([GRID_ID_COL, "date"]).reset_index(drop=True)
    city_panels[city_name] = panel

    print(f"  ✓ Shape: {panel.shape}  |  "
          f"Dates: {panel['date'].min().date()} → {panel['date'].max().date()}  |  "
          f"Cells: {panel[GRID_ID_COL].nunique()}")
    print(f"  Columns: {list(panel.columns)}")

print("\n✅ Wide panels assembled.")

## Part 7 — Save Outputs

In [ ]:
for city_name, panel in city_panels.items():
    out_dir = DATA_ROOT / "processed" / city_name / "grid_panel"
    out_dir.mkdir(parents=True, exist_ok=True)

    print(f"\nSaving: {city_name}")

    # Serialise: convert categoricals back to string, date to string
    panel_out = panel.copy()
    panel_out["date"] = panel_out["date"].astype(str)
    for col in panel_out.select_dtypes(include="category").columns:
        panel_out[col] = panel_out[col].astype(str)

    # Save as gzip-compressed CSV — 60-80% smaller than plain CSV, no dtype issues
    csv_path = out_dir / f"{city_name}_grid_panel.csv.gz"
    panel_out.to_csv(csv_path, index=False, compression="gzip")
    size_mb = csv_path.stat().st_size / 1e6
    print(f"  Saved: {csv_path.name}  ({size_mb:.1f} MB)")

print("\n✅ All outputs saved.")
print("\nTo reload in downstream notebooks:")
print("  df = pd.read_csv('.../{city}_grid_panel.csv.gz')  # pandas reads gzip automatically")
print("  grid_geom = gpd.read_file('.../{city}_grid_geometry.gpkg')")
print("  gdf = grid_geom[['spatial_id', 'geometry']].merge(df, on='spatial_id')")

## Part 8 — Validation & QC

### 8.1 Per-variable missingness in the grid panel

In [ ]:
print(f"{'City':<15} {'Variable':<12} {'Total rows':>10} {'Missing':>10} {'Missing %':>10}")
print("-" * 60)

for city_name, panel in city_panels.items():
    for var in VARIABLES:
        if var.key not in panel.columns:
            continue
        total   = len(panel)
        missing = panel[var.key].isna().sum()
        pct     = 100 * missing / total if total else 0
        print(f"{city_name:<15} {var.key:<12} {total:>10,} {missing:>10,} {pct:>9.1f}%")

### 8.2 Date coverage per city

In [ ]:
for city_name, panel in city_panels.items():
    dates      = pd.to_datetime(panel["date"]).sort_values().unique()
    print(f"  {city_name}:")
    print(f"    First : {dates.min().date()}  |  Last: {dates.max().date()}")

### 8.3 Spatial coverage maps — time-averaged value per grid cell

In [ ]:
from shapely import wkt as shapely_wkt

for city_name, panel in city_panels.items():
    city_cfg    = next((c for c in CITIES if c["name"] == city_name), {})
    active_vars = [v for v in VARIABLES if v.key in panel.columns]
    if not active_vars:
        continue

    # Load geometry lookup for plotting
    geom_path = DATA_ROOT / "processed" / city_name / "grid_panel" / f"{city_name}_grid_geometry.gpkg"
    grid_geom = gpd.read_file(geom_path)

    cell_means = panel.groupby(GRID_ID_COL)[[v.key for v in active_vars]].mean().reset_index()
    gdf_mean   = grid_geom[[GRID_ID_COL, "geometry"]].merge(cell_means, on=GRID_ID_COL, how="left")

    n_vars = len(active_vars)
    fig, axes = plt.subplots(1, n_vars, figsize=(10 * n_vars, 8))
    if n_vars == 1:
        axes = [axes]
    fig.suptitle(f"{city_cfg.get('display_name', city_name)} — Time-averaged grid values", fontsize=13)

    for ax, var in zip(axes, active_vars):
        gdf_mean.plot(column=var.key, ax=ax, legend=True, cmap="viridis",
                      legend_kwds={"label": var.key, "orientation": "horizontal", "shrink": 0.6})
        ax.set_title(var.key, fontsize=11)
        ax.set_axis_off()

    plt.tight_layout()
    plt.show()

### 8.4 City-level time series — spatial mean over all grid cells

In [ ]:
for city_name, panel in city_panels.items():
    city_cfg    = next((c for c in CITIES if c["name"] == city_name), {})
    active_vars = [v for v in VARIABLES if v.key in panel.columns]
    if not active_vars:
        continue

    daily_mean = panel.groupby("date")[[v.key for v in active_vars]].mean()

    fig, axes = plt.subplots(len(active_vars), 1,
                             figsize=(14, 3 * len(active_vars)), sharex=True)
    if len(active_vars) == 1:
        axes = [axes]
    fig.suptitle(f"{city_cfg.get('display_name', city_name)} — Daily city-mean (all grid cells)",
                 fontsize=13)

    for ax, var in zip(axes, active_vars):
        ax.plot(daily_mean.index, daily_mean[var.key], linewidth=0.8, color="steelblue")
        ax.set_ylabel(var.key)
        ax.grid(True, alpha=0.3)

    axes[-1].xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
    axes[-1].xaxis.set_major_locator(mdates.MonthLocator(interval=2))
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    plt.show()

## Part 9 — Summary

In [ ]:
print("=" * 60)
print("NOTEBOOK 01b SUMMARY")
print("=" * 60)

for city_name, panel in city_panels.items():
    city_cfg = next((c for c in CITIES if c["name"] == city_name), {})
    out_dir  = DATA_ROOT / "processed" / city_name / "grid_panel"
    print(f"\n{city_cfg.get('display_name', city_name)}")
    print(f"  Rows       : {len(panel):,}")
    print(f"  Grid cells : {panel[GRID_ID_COL].nunique():,}")
    print(f"  Dates      : {panel['date'].nunique()}  "
          f"({panel['date'].min().date()} \u2192 {panel['date'].max().date()})")
    print(f"  Variables  : {[v.key for v in VARIABLES if v.key in panel.columns]}")
    print(f"  Output dir : {out_dir}")
    for fname in [f"{city_name}_grid_panel.csv.gz", f"{city_name}_grid_geometry.gpkg"]:
        p    = out_dir / fname
        size = f"{p.stat().st_size / 1e6:.1f} MB" if p.exists() else "not found"
        print(f"    {fname}  ({size})")

print("\n\u2192 Next step: 02_panel_assembly.ipynb")